In [2]:
import seaborn as sns
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.image as mpimg
from matplotlib.ticker import ScalarFormatter

sns.set()


api = wandb.Api()
task_name = "sapg_allegro_kuka_reorientation"
project = api.runs("naoki-shitanda/"+task_name)
metric = "successes/time"
metric_name = "Episode Rewards"
#target_step = 20 * 10**9
target_steps = [5e9, 10e9, 15e9, 20e9]

print(len(project))    
task_runs = {
    #"CPO(w/AdR)": [],
    "CPO2(wo/AdR)": [],
    "SAPG": [],
    "PPO": [],
    #"PBT": [],
}

# 各手法名がrun.tagに含まれているrunを抽出
for key in task_runs.keys():
    for run in project:
        if any(key in tag for tag in run.tags):
            task_runs[key].append(run)


print(task_runs)
for key in task_runs.keys():
    print(key,":", len(task_runs[key]))
    

86
{'CPO2(wo/AdR)': [<Run naoki-shitanda/sapg_allegro_kuka_reorientation/uid_00_0611-1-0.001-1-0.2-0-0-sa-noawac-5class_24576envs_mixed_expl_learn_param_lf_1p_07_07_14h10m40s_seed2 (failed)>, <Run naoki-shitanda/sapg_allegro_kuka_reorientation/uid_00_0611-1-0.001-1-0.2-0-0-sa-noawac-5class_24576envs_mixed_expl_learn_param_lf_1p_07_07_07h26m21s_seed3 (failed)>, <Run naoki-shitanda/sapg_allegro_kuka_reorientation/uid_00_0611-1-0.001-1-0.2-0-0-sa-noawac-5class_24576envs_mixed_expl_learn_param_lf_1p_07_07_07h25m56s_seed1 (failed)>, <Run naoki-shitanda/sapg_allegro_kuka_reorientation/uid_00_0611-1-0.001-1-0.2-0-0-sa-noawac-5class_24576envs_mixed_expl_learn_param_lf_1p_07_07_07h25m55s_seed0 (failed)>, <Run naoki-shitanda/sapg_allegro_kuka_reorientation/uid_00_0611-1-0.001-1-0.2-0-0-sa-noawac-5class_24576envs_mixed_expl_learn_param_lf_1p_17_06_17h27m29s_seed4 (failed)>], 'SAPG': [<Run naoki-shitanda/sapg_allegro_kuka_reorientation/uid_00_0418-SAPG-24576_24576envs_mixed_expl_learn_param_lf_1p_

In [3]:
print(task_runs["SAPG"][0].summary)
print(task_runs["PPO"][0].summary)
print(task_runs["CPO2(wo/AdR)"][0].summary)

{'_runtime': 26676, '_step': 8199, '_timestamp': 1746010725.430349, '_wandb': {'runtime': 26700}, 'auxiliary_stats/off_on_grad_similarity': 0, 'auxiliary_stats/off_on_relative_grad_norms': 0, 'auxiliary_stats/off_policy_contrib': {'_type': 'histogram'}, 'auxiliary_stats/on_policy_contrib': {'_type': 'histogram'}, 'closest_keypoint_max_dist/frame': 0.03077881410717964, 'closest_keypoint_max_dist/iter': 0.03077881410717964, 'closest_keypoint_max_dist/time': 0.03077881410717964, 'closest_keypoint_max_dist_max/frame': 0.8845613598823547, 'closest_keypoint_max_dist_max/iter': 0.8845613598823547, 'closest_keypoint_max_dist_max/time': 0.8845613598823547, 'closest_keypoint_max_dist_median/frame': 0.012375337071716784, 'closest_keypoint_max_dist_median/iter': 0.012375337071716784, 'closest_keypoint_max_dist_median/time': 0.012375337071716784, 'closest_keypoint_max_dist_per_block/block_0/frame': 0.03353707492351532, 'closest_keypoint_max_dist_per_block/block_0/iter': 0.03353707492351532, 'closes

In [4]:
import torch
import numpy as np
import scipy.stats as stats
from scipy.stats import ttest_ind

# =========================
# データ抽出＋補間
# =========================

def extract_scores_at_step_interpolated(all_runs_dict, target_step, metric="rewards/step"):
    score_rows = []
    method_names = []
    all_seed_labels = []

    for method_name, runs in all_runs_dict.items():
        seed_scores = []
        seed_labels = []

        seeds = [run.config.get("seed", "not found") for run in runs]
        seed_set = set(seeds)

        for seed in seed_set:
            seed_runs = [run for run in runs if run.config.get("seed") == seed]
            seed_runs = sorted(seed_runs, key=lambda run: run.created_at)
            dfs = []

            for i, run in enumerate(seed_runs):
                df = run.history(keys=[metric, "global_step"])
                df = df.set_index("global_step")
                df = df.dropna()
                if i > 0:
                    df = df.iloc[40:]
                dfs.append(df)

            if len(dfs) == 0:
                continue

            seed_df = pd.concat(dfs, axis=0)
            seed_df = seed_df.rename(columns={metric: f"seed{seed}"})
            if "_step" in seed_df.columns:
                seed_df = seed_df.drop("_step", axis=1)

            seed_df = seed_df.sort_index()
            seed_df = seed_df.interpolate(method="linear", limit_direction="both")

            try:
                value = seed_df.loc[target_step].values[0]
            except KeyError:
                all_steps = seed_df.index.to_numpy()
                if target_step < all_steps[0] or target_step > all_steps[-1]:
                    continue
                lower_idx = max(i for i in range(len(all_steps)) if all_steps[i] <= target_step)
                upper_idx = min(i for i in range(len(all_steps)) if all_steps[i] >= target_step)
                lower = all_steps[lower_idx]
                upper = all_steps[upper_idx]
                val_lower = seed_df.loc[lower].values[0]
                val_upper = seed_df.loc[upper].values[0]
                weight = (target_step - lower) / (upper - lower + 1e-8)
                value = (1 - weight) * val_lower + weight * val_upper

            seed_scores.append(value)
            seed_labels.append(seed)

        if seed_scores:
            score_rows.append(seed_scores)
            method_names.append(method_name)
            all_seed_labels.append(seed_labels)

    max_seeds = max(len(row) for row in score_rows)
    padded = [row + [float('nan')] * (max_seeds - len(row)) for row in score_rows]
    tensor = torch.tensor(padded, dtype=torch.float32)
    return tensor, method_names, all_seed_labels

# =========================
# 信頼区間の計算
# =========================

def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = data.mean().item()
    std = data.std(unbiased=True).item()
    h = stats.t.ppf((1 + confidence) / 2., n - 1) * std / np.sqrt(n)
    return mean, h

# =========================
# メイン出力処理
# =========================

def evaluate_and_report(all_runs_dict, target_step, metric_name, task_name=""):
    tensor, methods, seeds = extract_scores_at_step_interpolated(all_runs_dict, target_step, metric=metric_name)

    with open("output.txt", "w") as f:
        def print_both(text):  # inner function
            print(text)
            f.write(text + "\n")

        print_both("==========================")
        print_both(f"Task: {task_name}")
        print_both(f"Target Env Step: {target_step}")
        print_both(f"Metric: {metric_name}")
        print_both("--------------------------\n")

        ci_list = []
        for i, method in enumerate(methods):
            mean, h = confidence_interval(tensor[i])
            ci_list.append((method, mean, h, tensor[i].numpy()))
            print_both(f"{method}: {tensor[i].numpy()}")
            print_both(f"seeds: {seeds[i]}")
            print_both(f"mean: {mean:.3f}")
            print_both(f"std: {tensor[i].std(unbiased=True).item():.3f}")
            print_both("--")
            #print_both(f"95% CI: [{mean - h:.3f}, {mean + h:.3f}]\n")

        ci_list_sorted = sorted(ci_list, key=lambda x: x[1], reverse=True)
        top_method = ci_list_sorted[0][0]
        top_mean, top_h, top_scores = ci_list_sorted[0][1], ci_list_sorted[0][2], ci_list_sorted[0][3]
        top_ci_lower = top_mean - top_h

        p_values = {}
        for method, mean, h, scores in ci_list:
            if method == top_method:
                continue
            _, p = ttest_ind(top_scores, scores, equal_var=False)
            p_values[method] = p

        top_tied_methods = [top_method] + [m for m in p_values if p_values[m] >= 0.05]
        sig_worse = [m for m in p_values if m not in top_tied_methods]

        print_both("==========================")
        print_both(f"✅ Top method by mean: {top_method}")
        print_both(f"Mean: {top_mean:.3f}, 95% CI: [{top_mean - top_h:.3f}, {top_mean + top_h:.3f}]\n")

        print_both("------ p-values vs top method (Welch's t-test) ------")
        for method, p in p_values.items():
            print_both(f"{method:15s}: p = {p:.4f}")

        print_both("\n------ Summary ------")
        if len(top_tied_methods) == 1:
            print_both(f"✅ Only '{top_method}' is significantly better than all others (p < 0.05)")
        else:
            print_both(f"✅ Statistically tied top methods (p ≥ 0.05 vs top): {', '.join(top_tied_methods)}")

        if sig_worse:
            print_both(f"⚠️  Methods significantly worse than '{top_method}' (p < 0.05): {', '.join(sig_worse)}")
        else:
            print_both(f"✅ No methods are significantly worse than '{top_method}'")

        print_both("\n--------------------------------------------------")
        print_both(f"Top: {top_method}")
        tied_others = [m for m in top_tied_methods if m != top_method]
        if tied_others:
            print_both(f"No Significant Difference: {', '.join(tied_others)}")
        else:
            print_both("No Significant Difference: (none)")

        if sig_worse:
            print_both(f"Significantly Worse: {', '.join(sig_worse)}")
        else:
            print_both("Significantly Worse: (none)")
        print_both("--------------------------------------------------")


In [5]:
for target_step in target_steps:
    print(f"Evaluating for target step: {target_step}")
    evaluate_and_report(
        all_runs_dict=task_runs,
        target_step=target_step,
        metric_name=metric,
        task_name=task_name,
    )


Evaluating for target step: 5000000000.0


Task: sapg_allegro_kuka_reorientation
Target Env Step: 5000000000.0
Metric: successes/time
--------------------------

CPO2(wo/AdR): [21.547195 31.76348  24.762142 12.525851 34.653557]
seeds: [0, 1, 2, 3, 4]
mean: 25.050
std: 8.754
--
SAPG: [ 5.0577464 17.816687   9.747575  14.871231   6.0656834]
seeds: [0, 1, 2, 3, 4]
mean: 10.712
std: 5.529
--
PPO: [0.44153738 0.19371402 0.5269123  1.0041885  0.37238547]
seeds: [0, 1, 2, 3, 4]
mean: 0.508
std: 0.303
--
✅ Top method by mean: CPO2(wo/AdR)
Mean: 25.050, 95% CI: [14.181, 35.920]

------ p-values vs top method (Welch's t-test) ------
SAPG           : p = 0.0182
PPO            : p = 0.0033

------ Summary ------
✅ Only 'CPO2(wo/AdR)' is significantly better than all others (p < 0.05)
⚠️  Methods significantly worse than 'CPO2(wo/AdR)' (p < 0.05): SAPG, PPO

--------------------------------------------------
Top: CPO2(wo/AdR)
No Significant Difference: (none)
Significantly Worse: SAPG, PPO
--------------------------------------------------


wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=9 sec). Create a new API with an integer timeout larger than 9, e.g., `api = wandb.Api(timeout=19)` to increase the graphql timeout.


Task: sapg_allegro_kuka_reorientation
Target Env Step: 15000000000.0
Metric: successes/time
--------------------------

CPO2(wo/AdR): [41.827282 38.32218  39.078682 37.157677 43.088707]
seeds: [0, 1, 2, 3, 4]
mean: 39.895
std: 2.478
--
SAPG: [36.741512 36.62571  33.22588  36.57463  35.519306]
seeds: [0, 1, 2, 3, 4]
mean: 35.737
std: 1.488
--
PPO: [1.0301484  1.9647996  0.1490609  0.72159004 0.05532868]
seeds: [0, 1, 2, 3, 4]
mean: 0.784
std: 0.773
--
✅ Top method by mean: CPO2(wo/AdR)
Mean: 39.895, 95% CI: [36.818, 42.972]

------ p-values vs top method (Welch's t-test) ------
SAPG           : p = 0.0161
PPO            : p = 0.0000

------ Summary ------
✅ Only 'CPO2(wo/AdR)' is significantly better than all others (p < 0.05)
⚠️  Methods significantly worse than 'CPO2(wo/AdR)' (p < 0.05): SAPG, PPO

--------------------------------------------------
Top: CPO2(wo/AdR)
No Significant Difference: (none)
Significantly Worse: SAPG, PPO
--------------------------------------------------
Eval